In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ── Raíz dinámica (carpeta donde corres el script/notebook) ────────────────
BASE_DIR = Path.cwd()

# Si tu proyecto tiene una carpeta raíz (ej: Data_SC) dentro de BASE_DIR:
ROOT = BASE_DIR

# ── Cargar .env ────────────────────────────────────────────────────────────
RUTA_ENV = ROOT / ".env"
load_dotenv(dotenv_path=RUTA_ENV)

# ── Rutas ──────────────────────────────────────────────────────────────────
RUTA_TXT = ROOT / "04TXTout"
RUTA_OUT = ROOT / "05Metadata"
RUTA_OUT.mkdir(parents=True, exist_ok=True)

RUTA_JSONL = RUTA_OUT / "metadata.jsonl"
RUTA_LOG   = RUTA_OUT / "log_metadata.csv"

# ── API ────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

MODELO = "gpt-4o-mini"
MAX_CHARS_CONTEXTO = 12000
REESCRIBIR = False

print("Configuración OK")
print(f"Root detectado: {ROOT}")
print(f"TXTs en:  {RUTA_TXT}")
print(f"Salida en: {RUTA_OUT}")


Configuración OK
Root detectado: c:\Users\DeSanto\Desktop\21 Sonda\Plataforma Marco\shami-sondabase\sondabase_v2
TXTs en:  c:\Users\DeSanto\Desktop\21 Sonda\Plataforma Marco\shami-sondabase\sondabase_v2\04TXTout
Salida en: c:\Users\DeSanto\Desktop\21 Sonda\Plataforma Marco\shami-sondabase\sondabase_v2\05Metadata


In [2]:
import json
import csv
import re
from datetime import datetime
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

AÑO_ACTUAL = datetime.now().year

def leer_txt(ruta: Path, max_chars: int = MAX_CHARS_CONTEXTO) -> str:
    try:
        texto = ruta.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        texto = ruta.read_text(encoding="latin-1")
    return texto.strip()[:max_chars]

def cargar_ids_procesados() -> set:
    """Lee el JSONL existente y devuelve el set de document_ids ya procesados."""
    ids = set()
    if RUTA_JSONL.exists():
        with open(RUTA_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    ids.add(obj.get("document_id", ""))
                except Exception:
                    pass
    return ids

def guardar_registro(registro: dict):
    with open(RUTA_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")

def guardar_log(row: list):
    escribir_header = not RUTA_LOG.exists()
    with open(RUTA_LOG, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        if escribir_header:
            writer.writerow(["fecha_hora", "archivo", "document_id", "estado", "detalle"])
        writer.writerow(row)

print("Funciones auxiliares OK")

Funciones auxiliares OK


In [4]:
PROMPT_SISTEMA = """
Eres un asistente experto en análisis de documentos corporativos de empresas de tecnología.
Extraes metadatos estructurados de documentos relacionados con licitaciones, contratos,
certificados de experiencia y referencias comerciales.
Siempre respondes SOLO con JSON válido, sin markdown ni texto extra.
""".strip()

PROMPT_EXTRACCION = """
Analiza el siguiente documento y extrae los metadatos en formato JSON con EXACTAMENTE esta estructura.
Si un campo no se puede determinar con certeza, usa null. No inventes datos.

{{
  "doc_type": uno de ["certificado_experiencia", "contrato", "adenda", "carta_referencia",
                       "constancia_conformidad", "certificacion_comercial", "excel_datos", "otro"],
  "client": "nombre del cliente o entidad emisora (string o null)",
  "country": "país en español, ej: Chile, Panamá, Brasil (string o null)",
  "project_domain": ["lista de dominios aplicables de: recaudo, flotas, semaforos, peajes,
                      iluminacion, movilidad, cloud, ti, medios_de_pago, gestion_municipal,
                      ambiental, otro"],
  "is_apostilled": true/false/null,
  "year": año como entero (ej: 2021) o null,
  "contract_value_usd": valor en USD como número (ej: 5000000) o null,
  "contract_duration_years": duración del contrato en años como número (ej: 3.5) o null,
  "technologies": ["lista de tecnologías mencionadas, ej: EMV, contactless, GTFS, etc."],
  "units_deployed": número entero de unidades instaladas/desplegadas si se menciona o null,
  "summary_one_line": "una sola oración que resuma de qué trata el documento",
  "language": uno de ["es", "pt", "en", "otro"],
  "validity_alert": true si el documento tiene más de 3 años o parece desactualizado, false si no
}}

Nombre del archivo: {nombre_archivo}

Contenido del documento:
{texto}
""".strip()

def extraer_metadata_llm(texto: str, nombre_archivo: str) -> dict:
    prompt = PROMPT_EXTRACCION.format(
        nombre_archivo=nombre_archivo,
        texto=texto
    )
    
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user",   "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)

def metadata_fallback(nombre_archivo: str) -> dict:
    """Metadata mínima cuando el LLM falla."""
    return {
        "doc_type": "otro",
        "client": None,
        "country": None,
        "project_domain": [],
        "is_apostilled": None,
        "year": None,
        "contract_value_usd": None,
        "contract_duration_years": None,
        "technologies": [],
        "units_deployed": None,
        "summary_one_line": f"No se pudo procesar: {nombre_archivo}",
        "language": None,
        "validity_alert": False
    }

print("Prompt y funciones de extracción OK")

Prompt y funciones de extracción OK


In [5]:
from tqdm import tqdm
import time

# Buscar todos los TXTs recursivamente
archivos_txt = sorted(RUTA_TXT.rglob("*.txt"))
print(f"TXTs encontrados: {len(archivos_txt)}")

# IDs ya procesados (para no reprocesar)
procesados = cargar_ids_procesados() if not REESCRIBIR else set()
print(f"Ya procesados: {len(procesados)}")

# Contadores
total = ok = omitidos = errores = 0

for ruta_txt in tqdm(archivos_txt, desc="Extrayendo metadata"):
    total += 1
    
    # document_id: ruta relativa sin extensión, separadores → __
    ruta_relativa = ruta_txt.relative_to(RUTA_TXT)
    document_id   = str(ruta_relativa.with_suffix("")).replace("\\", "__").replace("/", "__")
    
    # Saltar si ya fue procesado
    if document_id in procesados and not REESCRIBIR:
        omitidos += 1
        continue
    
    texto = leer_txt(ruta_txt)
    
    if not texto:
        guardar_log([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            str(ruta_txt), document_id, "omitido", "texto vacío"
        ])
        omitidos += 1
        continue
    
    try:
        metadata = extraer_metadata_llm(texto, ruta_txt.name)
        estado   = "ok"
        detalle  = metadata.get("doc_type", "?")
        
    except Exception as e:
        metadata = metadata_fallback(ruta_txt.name)
        estado   = "error_llm"
        detalle  = str(e)[:120]
        errores  += 1
        print(f"\n[ERROR] {ruta_txt.name}: {detalle}")
    
    # Calcular validity_alert aquí también por año
    year = metadata.get("year")
    if year and isinstance(year, int):
        metadata["validity_alert"] = (AÑO_ACTUAL - year) > 3
    
    # Registro completo
    registro = {
        "document_id":   document_id,
        "source_file":   ruta_txt.name,
        "relative_path": str(ruta_relativa),
        "source_path":   str(ruta_txt),
        "ingested_at":   datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **metadata
    }
    
    guardar_registro(registro)
    guardar_log([
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        str(ruta_txt), document_id, estado, detalle
    ])
    
    if estado == "ok":
        ok += 1
    
    # Pequeña pausa para no saturar la API en lotes grandes
    time.sleep(0.3)

print(f"\n{'='*50}")
print(f"Total procesados : {total}")
print(f"  OK             : {ok}")
print(f"  Omitidos       : {omitidos}")
print(f"  Errores LLM    : {errores}")
print(f"\nArchivo JSONL    : {RUTA_JSONL}")
print(f"Log CSV          : {RUTA_LOG}")

TXTs encontrados: 0
Ya procesados: 0


Extrayendo metadata: 0it [00:00, ?it/s]


Total procesados : 0
  OK             : 0
  Omitidos       : 0
  Errores LLM    : 0

Archivo JSONL    : c:\Users\DeSanto\Desktop\21 Sonda\Plataforma Marco\shami-sondabase\sondabase_v2\05Metadata\metadata.jsonl
Log CSV          : c:\Users\DeSanto\Desktop\21 Sonda\Plataforma Marco\shami-sondabase\sondabase_v2\05Metadata\log_metadata.csv


In [19]:
import pandas as pd

registros = []
with open(RUTA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        try:
            registros.append(json.loads(line))
        except Exception:
            pass

df = pd.DataFrame(registros)

columnas_vista = [
    "source_file", "doc_type", "client", "country",
    "is_apostilled", "year", "contract_value_usd",
    "units_deployed", "validity_alert", "summary_one_line"
]

# Mostrar solo las columnas que existen
columnas_disponibles = [c for c in columnas_vista if c in df.columns]
print(f"Documentos en metadata.jsonl: {len(df)}\n")
df[columnas_disponibles]

Documentos en metadata.jsonl: 89



,source_file,doc_type,client,country,is_apostilled,year,contract_value_usd,units_deployed,validity_alert,summary_one_line
0,1 - Certificado Metro Panamá- Sistema de Recau...,otro,None,None,None,NaN,NaN,NaN,False,No se pudo procesar: 1 - Certificado Metro Pan...
1,2 - Certificado Rapipass (Albrook) - Sistema d...,otro,None,None,None,NaN,NaN,NaN,False,No se pudo procesar: 2 - Certificado Rapipass ...
2,3 - Metro Valparaiso - Medios de Pago.txt,otro,None,None,None,NaN,NaN,NaN,False,No se pudo procesar: 3 - Metro Valparaiso - Me...
3,4 - Certificado TRANSANTIAGO 2012-2019.txt,otro,None,None,None,NaN,NaN,NaN,False,No se pudo procesar: 4 - Certificado TRANSANTI...
4,Aclaracion Certificado MIN TRANSP PUB Y METROP...,otro,None,None,None,NaN,NaN,NaN,False,No se pudo procesar: Aclaracion Certificado MI...
...,...,...,...,...,...,...,...,...,...,...
84,Referencia Transantiago AFT.txt,certificado_experiencia,Administrador Financiero del Transantiago,Chile,None,2019.0,NaN,5100.0,True,Certificación de experiencia de SONDA en la pr...
85,Referencia_ATTT may 2024.txt,certificado_experiencia,AUTORIDAD DE TRÁNSITO Y TRANSPORTE TERRESTRE D...,Panamá,None,2024.0,2.521819e+08,3000.0,False,Certificación de calidad y cumplimiento de con...
86,Referencia_Metro Panamá may 2024.txt,carta_referencia,"Metro de Panamá, S.A.",Panamá,None,2013.0,2.046342e+07,136.0,True,Certificación de Calidad y Cumplimiento de Con...
87,Transantiago 2019-2022 Apostillado (SIR SAE SI...,certificado_experiencia,Ministerio de Transporte y Telecomunicaciones ...,Chile,True,2022.0,1.233860e+05,6625.0,True,Certificado que acredita que SONDA S.A. ha cum...


In [20]:
print("=== Distribución por tipo de documento ===")
print(df["doc_type"].value_counts().to_string())

print("\n=== Distribución por país ===")
print(df["country"].value_counts().to_string())

print("\n=== Apostillados ===")
print(df["is_apostilled"].value_counts().to_string())

print("\n=== Alertas de vigencia (>3 años) ===")
print(df["validity_alert"].value_counts().to_string())

print("\n=== Documentos con valor de contrato informado ===")
con_valor = df[df["contract_value_usd"].notna()]
print(f"  {len(con_valor)} documentos")
if len(con_valor) > 0:
    print(con_valor[["source_file", "country", "contract_value_usd"]].to_string())

=== Distribución por tipo de documento ===
doc_type
certificado_experiencia    35
certificacion_comercial    31
contrato                   12
otro                        8
adenda                      2
carta_referencia            1

=== Distribución por país ===
country
Chile         39
Guatemala     12
Panamá         9
Colombia       8
Brasil         5
México         4
Costa Rica     2
Perú           2
Argentina      1
Ecuador        1

=== Apostillados ===
is_apostilled
True     12
False     8

=== Alertas de vigencia (>3 años) ===
validity_alert
True     61
False    28

=== Documentos con valor de contrato informado ===
  32 documentos
                                                                     source_file    country  contract_value_usd
6                                                  AFT 2005-2012 Apostillado.txt      Chile        7.200000e+07
9               Certificado - AFT (Proyecto Transantiago) - 09 20 - Original.txt      Chile        7.200000e+07
10               

In [22]:
# ─── INSTALACIÓN (ejecutar solo la primera vez) ───────────────────────────
# %pip install openai pandas

import sqlite3, json, pandas as pd
from pathlib import Path
from datetime import datetime

# ── CONFIG ────────────────────────────────────────────────────────────────
RUTA_METADATA = Path(r"C:\Users\lenovo\Desktop\Data_SC\05Metadata\metadata.jsonl")
RUTA_DB       = Path(r"C:\Users\lenovo\Desktop\Data_SC\05Metadata\documents.db")

# ── CREAR DB Y TABLA ──────────────────────────────────────────────────────
conn = sqlite3.connect(RUTA_DB)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA journal_mode=WAL")
conn.executescript("""
    CREATE TABLE IF NOT EXISTS documents (
        document_id             TEXT PRIMARY KEY,
        source_file             TEXT,
        relative_path           TEXT,
        source_path             TEXT,
        doc_type                TEXT,
        client                  TEXT,
        country                 TEXT,
        is_apostilled           INTEGER,
        year                    INTEGER,
        contract_value_usd      REAL,
        contract_duration_years REAL,
        units_deployed          INTEGER,
        summary_one_line        TEXT,
        language                TEXT,
        validity_alert          INTEGER,
        project_domain_json     TEXT,
        technologies_json       TEXT,
        ingested_at             TEXT
    );
    CREATE INDEX IF NOT EXISTS idx_country    ON documents(country);
    CREATE INDEX IF NOT EXISTS idx_doc_type   ON documents(doc_type);
    CREATE INDEX IF NOT EXISTS idx_apostilled ON documents(is_apostilled);
    CREATE INDEX IF NOT EXISTS idx_year       ON documents(year);
    CREATE INDEX IF NOT EXISTS idx_validity   ON documents(validity_alert);
    CREATE INDEX IF NOT EXISTS idx_client     ON documents(client);
""")
conn.commit()

# ── CARGAR JSONL ──────────────────────────────────────────────────────────
def safe_bool(v):
    if v is True:  return 1
    if v is False: return 0
    return None

def safe_int(v):
    try:    return int(v)
    except: return None

def safe_float(v):
    try:    return float(v)
    except: return None

ids_existentes = {r[0] for r in conn.execute("SELECT document_id FROM documents").fetchall()}
insertados = omitidos = errores = 0

with open(RUTA_METADATA, "r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()
        if not linea: continue
        try:
            obj = json.loads(linea)
        except:
            errores += 1
            continue

        doc_id = str(obj.get("document_id", "") or "")
        if not doc_id: continue

        if doc_id in ids_existentes:
            omitidos += 1
            continue

        pd_list = obj.get("project_domain", [])
        te_list = obj.get("technologies",   [])
        if not isinstance(pd_list, list): pd_list = []
        if not isinstance(te_list, list): te_list = []

        conn.execute("""
            INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            doc_id,
            str(obj.get("source_file",   "") or ""),
            str(obj.get("relative_path", "") or ""),
            str(obj.get("source_path",   "") or ""),
            str(obj.get("doc_type",      "") or ""),
            str(obj.get("client",        "") or "") or None,
            str(obj.get("country",       "") or "") or None,
            safe_bool (obj.get("is_apostilled")),
            safe_int  (obj.get("year")),
            safe_float(obj.get("contract_value_usd")),
            safe_float(obj.get("contract_duration_years")),
            safe_int  (obj.get("units_deployed")),
            str(obj.get("summary_one_line", "") or ""),
            str(obj.get("language",        "") or ""),
            safe_bool(obj.get("validity_alert", False)),
            json.dumps(pd_list, ensure_ascii=False),
            json.dumps(te_list, ensure_ascii=False),
            str(obj.get("ingested_at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))),
        ))
        insertados += 1

conn.commit()
print(f"Insertados: {insertados} | Omitidos: {omitidos} | Errores: {errores}")
print(f"DB guardada en: {RUTA_DB}")

# ── FUNCIÓN DE FILTRO ─────────────────────────────────────────────────────
def filter_documents(country=None, doc_type=None, is_apostilled=None,
                     year_min=None, year_max=None, project_domain=None,
                     include_outdated=True, value_min_usd=None, units_min=None):
    conditions, params = [], []
    if country:         conditions.append("LOWER(country) = LOWER(?)");  params.append(country)
    if doc_type:        conditions.append("LOWER(doc_type) = LOWER(?)"); params.append(doc_type)
    if is_apostilled is not None:
                        conditions.append("is_apostilled = ?");          params.append(1 if is_apostilled else 0)
    if year_min:        conditions.append("year >= ?");                   params.append(year_min)
    if year_max:        conditions.append("year <= ?");                   params.append(year_max)
    if not include_outdated:
                        conditions.append("(validity_alert = 0 OR validity_alert IS NULL)")
    if value_min_usd:   conditions.append("contract_value_usd >= ?");    params.append(value_min_usd)
    if units_min:       conditions.append("units_deployed >= ?");         params.append(units_min)
    if project_domain:
        conditions.append("EXISTS (SELECT 1 FROM json_each(project_domain_json) WHERE LOWER(value) = LOWER(?))")
        params.append(project_domain)

    where = ("WHERE " + " AND ".join(conditions)) if conditions else ""
    rows  = conn.execute(f"SELECT document_id FROM documents {where} ORDER BY year DESC", params).fetchall()
    return [r["document_id"] for r in rows]

print("filter_documents() lista ✓")

Insertados: 89 | Omitidos: 0 | Errores: 0
DB guardada en: C:\Users\lenovo\Desktop\Data_SC\05Metadata\documents.db
filter_documents() lista ✓


In [23]:
# ── ESTADÍSTICAS ──────────────────────────────────────────────────────────
total = conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print(f"Total documentos en DB: {total}\n")

for titulo, sql in [
    ("Por tipo",    "SELECT doc_type as valor, COUNT(*) n FROM documents GROUP BY doc_type ORDER BY n DESC"),
    ("Por país",    "SELECT country  as valor, COUNT(*) n FROM documents WHERE country IS NOT NULL GROUP BY country ORDER BY n DESC"),
    ("Apostillado", "SELECT CASE is_apostilled WHEN 1 THEN 'Sí' WHEN 0 THEN 'No' ELSE 'Desconocido' END valor, COUNT(*) n FROM documents GROUP BY is_apostilled"),
    ("Vigencia",    "SELECT CASE validity_alert WHEN 1 THEN 'Alerta >3 años' ELSE 'Vigente' END valor, COUNT(*) n FROM documents GROUP BY validity_alert"),
    ("Dominios",    "SELECT value valor, COUNT(*) n FROM documents, json_each(project_domain_json) GROUP BY value ORDER BY n DESC LIMIT 10"),
]:
    print(f"=== {titulo} ===")
    print(pd.read_sql(sql, conn).to_string(index=False))
    print()

# ── EJEMPLOS DE FILTRO ────────────────────────────────────────────────────
print("=== Ejemplos filter_documents() ===")
casos = [
    ("Certificados apostillados",          dict(is_apostilled=True, doc_type="certificado_experiencia")),
    ("Todo lo de Panamá",                  dict(country="Panamá")),
    ("Recaudo desde 2016",                 dict(project_domain="recaudo", year_min=2016)),
    ("Contratos > USD 5M",                 dict(value_min_usd=5_000_000)),
    ("Solo docs vigentes (<= 3 años)",     dict(include_outdated=False)),
]
for label, kwargs in casos:
    ids = filter_documents(**kwargs)
    print(f"  {label}: {len(ids)} docs → {ids[:3]}{'...' if len(ids)>3 else ''}")

Total documentos en DB: 89

=== Por tipo ===
                  valor  n
certificado_experiencia 35
certificacion_comercial 31
               contrato 12
                   otro  8
                 adenda  2
       carta_referencia  1

=== Por país ===
     valor  n
     Chile 39
 Guatemala 12
    Panamá  9
  Colombia  8
    Brasil  5
    México  4
Costa Rica  2
      Perú  2
 Argentina  1
   Ecuador  1

=== Apostillado ===
      valor  n
Desconocido 69
         No  8
         Sí 12

=== Vigencia ===
         valor  n
       Vigente 28
Alerta >3 años 61

=== Dominios ===
            valor  n
               ti 55
        movilidad 48
             otro 37
            cloud 21
          recaudo 18
   medios_de_pago 16
gestion_municipal 10
           flotas  8
        ambiental  3
           peajes  1

=== Ejemplos filter_documents() ===
  Certificados apostillados: 10 docs → ['Contratos__Brasil Recaudo__ATESTADO CAPACIDADE TÉCNICA MTU 29.11.2023 (apostilado) (1)', 'Contratos__Brasil Recaud

In [24]:
import os, re, json
from datetime import datetime
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────
RUTA_TXT_IN      = Path(r"C:\Users\lenovo\Desktop\Data_SC\04TXTout")
RUTA_CHUNKS_DIR  = Path(r"C:\Users\lenovo\Desktop\Data_SC\06Chunks")
RUTA_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
RUTA_CHUNKS_OUT  = RUTA_CHUNKS_DIR / "chunks.jsonl"
RUTA_METADATA    = Path(r"C:\Users\lenovo\Desktop\Data_SC\05Metadata\metadata.jsonl")

MAX_CHARS        = 2200
MIN_CHARS        = 250
OVERLAP          = 180
REESCRIBIR       = True

# ── CARGAR METADATA para enriquecer chunks ────────────────────────────────
meta_map = {}
with open(RUTA_METADATA, "r", encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
            meta_map[obj["document_id"]] = obj
        except: pass
print(f"Metadata cargada: {len(meta_map)} documentos")

# ── HELPERS ───────────────────────────────────────────────────────────────
def normalizar(texto):
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"===== PAGINA[ ]+(\d+)[ ]*=====", r"@@@PAGE_\1@@@", texto, flags=re.MULTILINE)
    texto = re.sub(r" *\n *", "\n", texto)
    texto = re.sub(r"(?<!\n)\n(?!\n)", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"@@@PAGE_(\d+)@@@", r"\n\n===== PAGINA \1 =====\n\n", texto)
    return re.sub(r"\n{3,}", "\n\n", texto).strip()

def partir_paginas(texto):
    partes  = re.split(r"(?=^===== PAGINA \d+ =====$)", texto, flags=re.MULTILINE)
    paginas = []
    num     = 1
    for parte in partes:
        parte = parte.strip()
        if not parte: continue
        m = re.match(r"^===== PAGINA (\d+) =====\n?", parte)
        if m:
            num = int(m.group(1))
            contenido = re.sub(r"^===== PAGINA \d+ =====\n?", "", parte, count=1).strip()
        else:
            contenido = parte.strip()
        if contenido:
            paginas.append({"page_num": num, "text": contenido})
    return paginas or [{"page_num": 1, "text": texto}]

def chunkar_pagina(text, max_c=MAX_CHARS, min_c=MIN_CHARS, overlap=OVERLAP):
    if len(text) <= max_c + 300:
        return [text] if len(text) >= min_c else []
    parrafos = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if not parrafos: parrafos = [text]
    chunks, actual = [], ""
    for p in parrafos:
        candidato = actual + "\n\n" + p if actual else p
        if len(candidato) <= max_c:
            actual = candidato
        else:
            if actual: chunks.append(actual.strip())
            cola = actual[-overlap:] if len(actual) > overlap else actual
            actual = (cola + "\n\n" + p) if cola else p
    if actual.strip(): chunks.append(actual.strip())
    # fusionar chunks muy cortos
    resultado = []
    for sc in chunks:
        if len(sc) < min_c and resultado:
            resultado[-1] += "\n\n" + sc
        else:
            resultado.append(sc)
    return [c for c in resultado if len(c) >= min_c]

# ── MAIN ──────────────────────────────────────────────────────────────────
if REESCRIBIR and RUTA_CHUNKS_OUT.exists():
    RUTA_CHUNKS_OUT.unlink()

archivos = sorted(RUTA_TXT_IN.rglob("*.txt"))
total_chunks = errores = 0

for ruta_txt in archivos:
    rel       = ruta_txt.relative_to(RUTA_TXT_IN)
    doc_id    = str(rel.with_suffix("")).replace("\\","__").replace("/","__")
    meta      = meta_map.get(doc_id, {})

    try:
        texto = ruta_txt.read_text(encoding="utf-8")
    except:
        texto = ruta_txt.read_text(encoding="latin-1")

    texto    = normalizar(texto)
    paginas  = partir_paginas(texto)
    ahora    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    idx      = 0

    for pag in paginas:
        subchunks = chunkar_pagina(pag["text"])
        for i, chunk in enumerate(subchunks):
            registro = {
                # ── identificación ──
                "chunk_id":      f"{ruta_txt.stem}__p{pag['page_num']}__c{idx}",
                "document_id":   doc_id,
                "source_file":   ruta_txt.name,
                "relative_path": str(rel),
                # ── posición ──
                "page_start":    pag["page_num"],
                "page_end":      pag["page_num"],
                "chunk_index":   idx,
                # ── texto ──
                "text":          chunk,
                "char_count":    len(chunk),
                # ── metadatos estructurados (del paso 1) ──
                "doc_type":      meta.get("doc_type"),
                "client":        meta.get("client"),
                "country":       meta.get("country"),
                "is_apostilled": meta.get("is_apostilled"),
                "year":          meta.get("year"),
                "project_domain": meta.get("project_domain", []),
                "validity_alert": meta.get("validity_alert"),
                "summary_one_line": meta.get("summary_one_line"),
                # ── ingesta ──
                "ingested_at":   ahora,
            }
            with open(RUTA_CHUNKS_OUT, "a", encoding="utf-8") as f:
                f.write(json.dumps(registro, ensure_ascii=False) + "\n")
            idx += 1
            total_chunks += 1

    print(f"[OK] {ruta_txt.name} → {idx} chunks")

print(f"\nTotal chunks generados: {total_chunks}")
print(f"Archivo: {RUTA_CHUNKS_OUT}")

Metadata cargada: 89 documentos
[OK] 1 - Certificado Metro Panamá- Sistema de Recaudo (Apostillado).txt → 2 chunks
[OK] 2 - Certificado Rapipass (Albrook) - Sistema de Recaudo (Apostillado).txt → 2 chunks
[OK] 3 - Metro Valparaiso - Medios de Pago.txt → 1 chunks
[OK] 4 - Certificado TRANSANTIAGO 2012-2019.txt → 3 chunks
[OK] Aclaracion Certificado MIN TRANSP PUB Y METROP del 10 mayo 2022.txt → 2 chunks
[OK] Adenda 5 Contrato 35-2011 ATTT.txt → 8 chunks
[OK] AFT 2005-2012 Apostillado.txt → 7 chunks
[OK] AFT 2005-2012.txt → 6 chunks
[OK] Certificación Rapipass SONDA.txt → 1 chunks
[OK] Certificado - AFT (Proyecto Transantiago) - 09 20 - Original.txt → 5 chunks
[OK] Certificado - Transantiago - 06 22 - Apostilhado.txt → 6 chunks
[OK] Certificado AFT Proyecto Transantiago (30-09-2020) a conformidad.txt → 5 chunks
[OK] Certificado AFT Transantiago Apostillado 23-06-2022 v2.txt → 7 chunks
[OK] Certificado AFT Transantiago Apostillado 23-06-2022.txt → 6 chunks
[OK] Certificado ATTT Panama cop

In [25]:
registros = []
with open(RUTA_CHUNKS_OUT, "r", encoding="utf-8") as f:
    for line in f:
        try: registros.append(json.loads(line))
        except: pass

import pandas as pd
df = pd.DataFrame(registros)
print(f"Total chunks: {len(df)}")
print(f"Documentos únicos: {df['document_id'].nunique()}\n")

print("=== Chunks por documento ===")
print(df.groupby("source_file")["chunk_id"].count().sort_values(ascending=False).to_string())

print("\n=== Muestra de un chunk con metadata enriquecida ===")
muestra = registros[0]
for k, v in muestra.items():
    if k != "text":
        print(f"  {k}: {v}")
print(f"  text (primeros 200 chars): {muestra['text'][:200]}...")

Total chunks: 696
Documentos únicos: 88

=== Chunks por documento ===
source_file
CONTRATO_DE_PRESTACIÓN_DE_SERVICIOS_DE_RECAUDO_GESTIÓN_Y_CONTROL_DE_FLOTA_17082023_FDO.txt    92
Contrato 7.txt                                                                                51
Guatemala20250227_Urbanorte firmado (1).txt                                                   48
Contrato Visiones para Servicios Firmado (2).txt                                              48
Contrato N6 (1).txt                                                                           48
Suministro y Montaje de equipos y software  VF 24.11.2021.txt                                 48
CONTRATO Firmado ambas partes[45]_compressed.txt                                              42
ACTA RPyH firmada por proponentes (1).txt                                                     32
Adenda visión para servicio firmada-2025 (1).txt                                              19
Cto. SN-008739 Empresa de los Ferrocarriles d

In [ ]:
# %pip install chromadb openai

import json, time
from pathlib import Path
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import chromadb

# ── CONFIG ────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")RUTA_CHUNKS     = Path(r"C:\Users\lenovo\Desktop\Data_SC\06Chunks\chunks.jsonl")
RUTA_METADATA   = Path(r"C:\Users\lenovo\Desktop\Data_SC\05Metadata\metadata.jsonl")
RUTA_CHROMA     = Path(r"C:\Users\lenovo\Desktop\Data_SC\08ChromaDB")
EMBED_MODEL     = "text-embedding-3-small"
BATCH_SIZE      = 50   # chunks por llamada a OpenAI

# ── INIT ──────────────────────────────────────────────────────────────────
ef = OpenAIEmbeddingFunction(api_key=OPENAI_API_KEY, model_name=EMBED_MODEL)

chroma = chromadb.PersistentClient(path=str(RUTA_CHROMA))

col_chunks = chroma.get_or_create_collection(name="chunks", embedding_function=ef,
                                              metadata={"hnsw:space": "cosine"})
col_docs   = chroma.get_or_create_collection(name="docs",   embedding_function=ef,
                                              metadata={"hnsw:space": "cosine"})

def safe_meta(val, default=""):
    """ChromaDB solo acepta str/int/float en metadata, no listas ni None."""
    if val is None:              return default
    if isinstance(val, list):    return ", ".join(str(x) for x in val)
    if isinstance(val, bool):    return int(val)
    return val

def add_batch(collection, ids, docs, metas):
    """Inserta en lotes y omite los ids que ya existen."""
    existentes = set(collection.get(ids=ids)["ids"])
    nuevos = [(i, d, m) for i, d, m in zip(ids, docs, metas) if i not in existentes]
    if not nuevos: return 0
    nids, ndocs, nmetas = zip(*nuevos)
    collection.add(ids=list(nids), documents=list(ndocs), metadatas=list(nmetas))
    return len(nids)

# ════════════════════════════════════════════════════════════════════════════
# COLECCIÓN 1 — CHUNKS  (búsqueda precisa, detalles, cláusulas, datos exactos)
# ════════════════════════════════════════════════════════════════════════════
print("Cargando colección CHUNKS...")

ids_b, docs_b, metas_b = [], [], []
total_chunks = 0

with open(RUTA_CHUNKS, "r", encoding="utf-8") as f:
    for linea in f:
        try: obj = json.loads(linea)
        except: continue

        ids_b.append(str(obj["chunk_id"]))
        docs_b.append(str(obj["text"]))
        metas_b.append({
            "document_id":    safe_meta(obj.get("document_id")),
            "source_file":    safe_meta(obj.get("source_file")),
            "page_start":     safe_meta(obj.get("page_start"), 0),
            "page_end":       safe_meta(obj.get("page_end"),   0),
            "chunk_index":    safe_meta(obj.get("chunk_index"), 0),
            # ── filtros estructurados ──
            "doc_type":       safe_meta(obj.get("doc_type")),
            "client":         safe_meta(obj.get("client")),
            "country":        safe_meta(obj.get("country")),
            "is_apostilled":  safe_meta(obj.get("is_apostilled"), -1),
            "year":           safe_meta(obj.get("year"), 0),
            "project_domain": safe_meta(obj.get("project_domain")),
            "validity_alert": safe_meta(obj.get("validity_alert"), 0),
            "summary_one_line": safe_meta(obj.get("summary_one_line")),
        })

        if len(ids_b) == BATCH_SIZE:
            total_chunks += add_batch(col_chunks, ids_b, docs_b, metas_b)
            ids_b, docs_b, metas_b = [], [], []
            print(f"  chunks cargados: {total_chunks}", end="\r")
            time.sleep(0.5)

if ids_b:
    total_chunks += add_batch(col_chunks, ids_b, docs_b, metas_b)

print(f"\nCHUNKS listos → {col_chunks.count()} en ChromaDB")

# ════════════════════════════════════════════════════════════════════════════
# COLECCIÓN 2 — DOCS  (búsqueda global, panorama, ¿qué tenemos sobre X?)
# ════════════════════════════════════════════════════════════════════════════
print("\nCargando colección DOCS...")

ids_b, docs_b, metas_b = [], [], []
total_docs = 0

with open(RUTA_METADATA, "r", encoding="utf-8") as f:
    for linea in f:
        try: obj = json.loads(linea)
        except: continue

        doc_id = str(obj.get("document_id", ""))
        if not doc_id: continue

        # Texto enriquecido que se embebe (no solo el nombre del archivo)
        texto_embed = "\n".join(filter(None, [
            f"ARCHIVO: {obj.get('source_file','')}",
            f"TIPO: {obj.get('doc_type','')}",
            f"CLIENTE: {obj.get('client','')}",
            f"PAIS: {obj.get('country','')}",
            f"APOSTILLADO: {'Sí' if obj.get('is_apostilled') else 'No'}",
            f"AÑO: {obj.get('year','')}",
            f"DOMINIOS: {', '.join(obj.get('project_domain') or [])}",
            f"RESUMEN: {obj.get('summary_one_line','')}",
        ]))

        ids_b.append(doc_id)
        docs_b.append(texto_embed)
        metas_b.append({
            "document_id":    doc_id,
            "source_file":    safe_meta(obj.get("source_file")),
            "doc_type":       safe_meta(obj.get("doc_type")),
            "client":         safe_meta(obj.get("client")),
            "country":        safe_meta(obj.get("country")),
            "is_apostilled":  safe_meta(obj.get("is_apostilled"), -1),
            "year":           safe_meta(obj.get("year"), 0),
            "project_domain": safe_meta(obj.get("project_domain")),
            "validity_alert": safe_meta(obj.get("validity_alert"), 0),
            "summary_one_line": safe_meta(obj.get("summary_one_line")),
            "language":       safe_meta(obj.get("language")),
        })

        if len(ids_b) == BATCH_SIZE:
            total_docs += add_batch(col_docs, ids_b, docs_b, metas_b)
            ids_b, docs_b, metas_b = [], [], []
            time.sleep(0.5)

if ids_b:
    total_docs += add_batch(col_docs, ids_b, docs_b, metas_b)

print(f"DOCS listos → {col_docs.count()} en ChromaDB")
print(f"\nChromaDB guardada en: {RUTA_CHROMA}")

Cargando colección CHUNKS...
  chunks cargados: 650
CHUNKS listos → 696 en ChromaDB

Cargando colección DOCS...
DOCS listos → 89 en ChromaDB

ChromaDB guardada en: C:\Users\lenovo\Desktop\Data_SC\08ChromaDB


In [28]:
# ── Test 1: búsqueda semántica libre ─────────────────────────────────────
print("=== Búsqueda libre: 'sistema de recaudo transporte público' ===")
r = col_chunks.query(query_texts=["sistema de recaudo transporte público"], n_results=3,
                     include=["documents","metadatas","distances"])
for i in range(len(r["ids"][0])):
    m = r["metadatas"][0][i]
    print(f"\n  [{i+1}] {m['source_file']} | {m['country']} | año {m['year']}")
    print(f"       Dist: {r['distances'][0][i]:.4f}")
    print(f"       {r['documents'][0][i][:150]}...")

# ── Test 2: búsqueda CON pre-filtro (la mejora clave) ────────────────────
print("\n\n=== Búsqueda filtrada: solo apostillados ===")
r = col_chunks.query(
    query_texts=["experiencia en sistemas de recaudo"],
    n_results=3,
    where={"is_apostilled": 1},           # ← filtro duro antes de buscar
    include=["documents","metadatas","distances"]
)
for i in range(len(r["ids"][0])):
    m = r["metadatas"][0][i]
    print(f"\n  [{i+1}] {m['source_file']} | apostillado={m['is_apostilled']} | {m['country']}")
    print(f"       {r['documents'][0][i][:150]}...")

# ── Test 3: búsqueda a nivel de documentos (¿qué tenemos sobre X?) ────────
print("\n\n=== Docs: ¿qué referencias tenemos en Panamá? ===")
r = col_docs.query(
    query_texts=["referencias Panamá"],
    n_results=5,
    where={"country": "Panamá"},
    include=["documents","metadatas","distances"]
)
for i in range(len(r["ids"][0])):
    m = r["metadatas"][0][i]
    print(f"\n  [{i+1}] {m['source_file']}")
    print(f"       {m['summary_one_line']}")

print("\n✓ ChromaDB operativa con filtros estructurados")

=== Búsqueda libre: 'sistema de recaudo transporte público' ===

  [1] CONTRATO_DE_PRESTACIÓN_DE_SERVICIOS_DE_RECAUDO_GESTIÓN_Y_CONTROL_DE_FLOTA_17082023_FDO.txt | Colombia | año 2023
       Dist: 0.3554
       [METODO: texto]

26 entidades bancarias que pueden utilizarse para validar el pago e ingreso al Sistema de transporte público. Usualmente el sector fi...

  [2] Certificado Trancaribe v3 06-02-2024-1 - 06.05.2025.txt | Colombia | año 2023
       Dist: 0.4079
       [METODO: texto]

Parágrafo 1. Para dar inicio a la Subetapa Operativa, el Sistema de Recaudo deberá estar implementado y funcionando completamente de ...

  [3] 20250317 Certificado Trancaribe v3 06-02-2024-1 - 17.03.2025.txt | Colombia | año 2025
       Dist: 0.4079
       [METODO: texto]

Parágrafo 1. Para dar inicio a la Subetapa Operativa, el Sistema de Recaudo deberá estar implementado y funcionando completamente de ...


=== Búsqueda filtrada: solo apostillados ===

  [1] Certificado GUATEMALA Apostillado 23-06-

In [ ]:
import json, re, sqlite3
from pathlib import Path
from openai import OpenAI
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# ── CONFIG ────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
RUTA_CHROMA    = Path(r"C:\Users\lenovo\Desktop\Data_SC\08ChromaDB")
RUTA_DB        = Path(r"C:\Users\lenovo\Desktop\Data_SC\05Metadata\documents.db")
MODELO_CHAT    = "gpt-4o-mini"
EMBED_MODEL    = "text-embedding-3-small"

llm    = OpenAI(api_key=OPENAI_API_KEY)
ef     = OpenAIEmbeddingFunction(api_key=OPENAI_API_KEY, model_name=EMBED_MODEL)
chroma = chromadb.PersistentClient(path=str(RUTA_CHROMA))

col_chunks = chroma.get_collection("chunks", embedding_function=ef)
col_docs   = chroma.get_collection("docs",   embedding_function=ef)

db = sqlite3.connect(str(RUTA_DB))
db.row_factory = sqlite3.Row

# ── ROUTER ────────────────────────────────────────────────────────────────
def detectar_tipo(pregunta: str) -> str:
    q = pregunta.lower()
    if any(x in q for x in ["qué documentos","qué referencias","qué contratos",
                              "qué tenemos","listado","lista","todos los","cuáles"]):
        return "doc"
    if any(x in q for x in ["cláusula","clausula","valor exacto","monto","fecha exacta",
                              "artículo","folio","rut","nit","iva","número de contrato"]):
        return "chunk"
    if any(x in q for x in ["apostillado","vigente","país","pais","cliente específico"]):
        return "chunk"
    # fallback LLM
    r = llm.chat.completions.create(
        model=MODELO_CHAT, temperature=0,
        messages=[{"role":"user","content":
            f"Clasifica en UNA palabra: doc (visión general/listado) o chunk (dato preciso/cláusula).\nQuery: {pregunta}"}]
    )
    t = r.choices[0].message.content.strip().lower()
    return "doc" if "doc" in t else "chunk"

# ── EXTRAER FILTROS DE LA PREGUNTA ────────────────────────────────────────
PAISES = ["chile","panamá","panama","brasil","brazil","colombia","guatemala",
          "méxico","mexico","uruguay","el salvador","perú","peru","argentina"]

def extraer_filtros(pregunta: str) -> dict:
    q = pregunta.lower()
    filtros = {}
    for p in PAISES:
        if p in q:
            filtros["country"] = p.capitalize().replace("Panama","Panamá").replace("Mexico","México").replace("Brazil","Brasil")
            break
    if any(x in q for x in ["apostillado","apostilled","internacional"]):
        filtros["is_apostilled"] = 1
    return filtros

# ── BÚSQUEDA EN CHROMADB ──────────────────────────────────────────────────
def buscar(pregunta: str, tipo: str, filtros: dict, n=6) -> list[dict]:
    col    = col_docs if tipo == "doc" else col_chunks
    kwargs = dict(query_texts=[pregunta], n_results=n,
                  include=["documents","metadatas","distances"])
    if filtros:
        kwargs["where"] = filtros

    try:
        r = col.query(**kwargs)
    except Exception:
        # si el filtro no da resultados, buscar sin él
        r = col.query(query_texts=[pregunta], n_results=n,
                      include=["documents","metadatas","distances"])

    items = []
    for i in range(len(r["ids"][0])):
        items.append({
            "id":       r["ids"][0][i],
            "text":     r["documents"][0][i],
            "meta":     r["metadatas"][0][i],
            "distance": r["distances"][0][i],
        })
    return items

# ── CONSTRUIR CONTEXTO ────────────────────────────────────────────────────
def construir_contexto(items: list[dict]) -> str:
    bloques = []
    for it in items:
        m = it["meta"]
        encabezado = " | ".join(filter(None,[
            m.get("source_file",""),
            m.get("country",""),
            f"año {m.get('year','')}" if m.get("year") else None,
            "APOSTILLADO" if m.get("is_apostilled")==1 else None,
            f"p.{m.get('page_start','')}" if m.get("page_start") else None,
        ]))
        bloques.append(f"[{encabezado}]\n{it['text']}")
    return "\n\n".join(bloques)

# ── FUNCIÓN PRINCIPAL DE CONSULTA ─────────────────────────────────────────
def consultar(pregunta: str, pais: str = None, solo_apostillados: bool = False,
              año_desde: int = None) -> dict:
    """
    Parámetros opcionales:
      pais             → filtra por país (ej: "Panamá")
      solo_apostillados → solo documentos apostillados
      año_desde        → solo documentos desde ese año
    """
    tipo    = detectar_tipo(pregunta)
    filtros = extraer_filtros(pregunta)

    # Filtros manuales tienen prioridad
    if pais:              filtros["country"]       = pais
    if solo_apostillados: filtros["is_apostilled"] = 1

    # year no soporta where directo en chroma → lo aplicamos post-búsqueda
    items = buscar(pregunta, tipo, filtros, n=8)

    if año_desde:
        items = [x for x in items if (x["meta"].get("year") or 0) >= año_desde]

    if not items:
        return {"respuesta": "No encontré documentos que coincidan con tu búsqueda.", "fuentes": []}

    contexto = construir_contexto(items)

    prompt = f"""Eres SONDA Smart Cities Assistant.
Responde SOLO con la información del contexto. No inventes datos.
Si la información no está disponible, responde: "No encontré esa información en los documentos disponibles."

Reglas:
- Sé directo y conciso
- Siempre cita el nombre del archivo fuente
- Si hay múltiples resultados, usa una tabla con columnas: Cliente | País | Documento | Apostillado | Año
- Si el documento tiene más de 3 años, indica "(⚠ revisar vigencia)"
- Prioriza documentos apostillados para licitaciones internacionales

Pregunta: {pregunta}

Contexto documental:
{contexto}
"""
    r = llm.chat.completions.create(
        model=MODELO_CHAT, temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    fuentes = [{"archivo": x["meta"].get("source_file",""),
                "pais":    x["meta"].get("country",""),
                "año":     x["meta"].get("year",""),
                "apostillado": x["meta"].get("is_apostilled")==1,
                "distancia": round(x["distance"],4)} for x in items]

    return {"tipo": tipo, "filtros": filtros,
            "respuesta": r.choices[0].message.content,
            "fuentes": fuentes}

# ── FUNCIÓN RFP: MATRIZ DE CUMPLIMIENTO ───────────────────────────────────
def verificar_rfp(texto_rfp: str) -> None:
    """
    Recibe el texto de requisitos de una licitación y genera
    una matriz de cumplimiento: CUMPLE / PARCIAL / NO ENCONTRADO
    """
    # 1. Descomponer requisitos
    r = llm.chat.completions.create(
        model=MODELO_CHAT, temperature=0,
        messages=[{"role":"user","content":f"""
Extrae los requisitos habilitantes de esta licitación como lista JSON.
Cada elemento debe tener: {{"id":"R1","texto":"descripción corta","dominio":"área técnica"}}
Devuelve SOLO el JSON array, sin markdown.

Licitación:
{texto_rfp}
"""}])
    try:
        requisitos = json.loads(r.choices[0].message.content.strip())
    except:
        print("No se pudo parsear los requisitos"); return

    print(f"Requisitos detectados: {len(requisitos)}\n")
    print(f"{'ID':<5} {'Requisito':<55} {'Estado':<12} {'Evidencia'}")
    print("─"*120)

    for req in requisitos:
        items = buscar(req["texto"], tipo="chunk", filtros={}, n=4)
        if not items:
            print(f"{req['id']:<5} {req['texto'][:53]:<55} {'NO ENCONTRADO':<12}")
            continue

        contexto = construir_contexto(items)

        eval_r = llm.chat.completions.create(
            model=MODELO_CHAT, temperature=0,
            messages=[{"role":"user","content":f"""
Evalúa si el contexto documental demuestra que se cumple este requisito.
Responde SOLO con JSON: {{"estado":"CUMPLE|PARCIAL|NO_CUMPLE","evidencia":"descripción breve en max 80 chars"}}

Requisito: {req["texto"]}

Contexto:
{contexto}
"""}])
        try:
            ev = json.loads(eval_r.choices[0].message.content.strip())
            estado   = ev.get("estado","?")
            evidencia = ev.get("evidencia","")[:75]
        except:
            estado, evidencia = "?", ""

        emoji = {"CUMPLE":"✅","PARCIAL":"⚠️","NO_CUMPLE":"❌"}.get(estado,"❓")
        print(f"{req['id']:<5} {req['texto'][:53]:<55} {emoji+' '+estado:<12} {evidencia}")

print("Motor de consultas listo ✓")

Motor de consultas listo ✓


In [32]:
# ── PRUEBA 1: pregunta libre ───────────────────────────────────────────────
r = consultar("¿Qué experiencia tenemos en sistemas de recaudo?")
print(r["respuesta"])
print(f"\n[tipo={r['tipo']} | filtros={r['filtros']}]")

print("\n" + "═"*80 + "\n")

# ── PRUEBA 2: con filtros manuales ────────────────────────────────────────
r = consultar("referencias de proyectos ejecutados", pais="Panamá", solo_apostillados=True)
print(r["respuesta"])

print("\n" + "═"*80 + "\n")

# ── PRUEBA 3: RFP real — pega aquí los requisitos de una licitación ───────
rfp = """
Experiencia en proyectos de iluminación pública:
Contar con al menos 5 años de experiencia operando en mercados relacionados
con proyectos de iluminación en vías públicas o alumbrado público.

Suministro e instalación de luminarias LED:
Haber suministrado e instalado al menos 70,000 luminarias de tecnología LED
para alumbrado público en forma acumulada en los últimos 10 años.

Sistemas de medición avanzada y telegestión:
Contar con al menos 5 años de experiencia en suministro e instalación de
sistemas de medición avanzada (AMI) y dispositivos de telegestión.
"""

print("=== MATRIZ DE CUMPLIMIENTO RFP ===\n")
verificar_rfp(rfp)

Aquí tienes la información sobre la experiencia de SONDA en sistemas de recaudo:

| Cliente         | País     | Documento                                                                 | Apostillado | Año  |
|------------------|----------|---------------------------------------------------------------------------|-------------|------|
| TRANSCARIBE S.A. | Colombia | Certificado Trancaribe v3 06-02-2024-1 - 06.05.2025.txt                 | No          | 2023 |
| TRANSCARIBE S.A. | Colombia | 20250317 Certificado Trancaribe v3 06-02-2024-1 - 17.03.2025.txt        | No          | 2025 |
| No especificado   | Colombia | CONTRATO_DE_PRESTACIÓN_DE_SERVICIOS_DE_RECAUDO_GESTIÓN_Y_CONTROL_DE_FLOTA_17082023_FDO.txt | No          | 2023 |
| No especificado   | Panamá   | Referencia_Metro Panamá may 2024.txt                                     | No          | 2013 (⚠ revisar vigencia) |
| No especificado   | Chile    | Suministro y Montaje de equipos y software VF 24.11.2021.txt            | No 